# Phase 1 — 장미 잎 상태 분류 베이스라인 (YOLO11 vs YOLO26, V100)

1. 환경 확인 / 설치
2. 데이터 준비: 원본 파일명 기준 분할(증강 누수 방지) + RGBA→RGB 배경 합성 + 소수 클래스 오버샘플링
3. `yolo11s-cls` vs `yolo26s-cls` 학습
4. 테스트셋 평가 (클래스별 recall / macro-F1 / 혼동행렬)

> `SRC_DIR` 에 `Black Spot Leaves / Fresh Leaves / Hole Leaves / Yellow Leaves` 폴더가 있는 `Dataset` 경로를 지정하세요.

In [ ]:
!nvidia-smi
# %pip install -q -U ultralytics scikit-learn   # 2026-10-13 비활성화: 이 설치가 서버 opencv 4.7/4.11 충돌의 원인 (README 참고)

In [ ]:
import sys; sys.path.insert(0, '../common'); import paths  # 공용 경로 + opencv 충돌 우회 (cv2/ultralytics보다 먼저)
import torch, ultralytics
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0))
print('VRAM %.1f GB' % (torch.cuda.get_device_properties(0).total_memory / 1e9))
print('ultralytics', ultralytics.__version__)

## 2. 데이터 준비

In [ ]:
import os, re, random, shutil
from collections import defaultdict, Counter
from pathlib import Path
import numpy as np
from PIL import Image

SRC_DIR = Path('../common/Dataset')  # 원본 데이터 (작업 디렉토리: model/phase1_cls)
OUT_DIR = Path('data/rose_cls')      # YOLO cls 형식 출력 경로
SEED = 42
SPLIT = (0.70, 0.15, 0.15)           # train / val / test
TARGET_PER_CLASS = 1400              # train 클래스당 목표 장수(오버샘플링 시 배경을 바꿔 다양화)
MAX_SIDE = 640                       # 저장 해상도 상한 (학습은 224~320)

random.seed(SEED); np.random.seed(SEED)
CLASSES = sorted(d.name for d in SRC_DIR.iterdir() if d.is_dir())
print(CLASSES)

def stem_of(name):
    # '20241104_101153_aug_108.png' -> '20241104_101153'
    return re.split(r'_aug_', Path(name).stem)[0]

In [ ]:
# 원본 stem 단위로 묶어 분할 -> 같은 원본에서 나온 증강본이 val/test 로 새지 않음
split_files = {s: defaultdict(list) for s in ('train', 'val', 'test')}
for c in CLASSES:
    groups = defaultdict(list)
    for f in sorted((SRC_DIR / c).glob('*.png')):
        groups[stem_of(f.name)].append(f)
    stems = sorted(groups); random.shuffle(stems)
    n = len(stems); n_tr = int(n * SPLIT[0]); n_va = int(n * SPLIT[1])
    parts = {'train': stems[:n_tr], 'val': stems[n_tr:n_tr + n_va], 'test': stems[n_tr + n_va:]}
    for s, ss in parts.items():
        for st in ss:
            files = groups[st]
            if s != 'train':  # 평가셋은 원본만 사용
                files = [f for f in files if '_aug_' not in f.name] or files[:1]
            split_files[s][c].extend(files)

for s in split_files:
    print(s, {c: len(v) for c, v in split_files[s].items()})

In [ ]:
def random_background(w, h, rng):
    kind = rng.integers(0, 4)
    if kind == 0:   # 단색
        return np.full((h, w, 3), rng.integers(0, 256, 3), np.uint8)
    if kind == 1:   # 그라디언트
        a, b = rng.integers(0, 256, 3), rng.integers(0, 256, 3)
        t = np.linspace(0, 1, w)[None, :, None]
        return np.broadcast_to(a * (1 - t) + b * t, (h, w, 3)).astype(np.uint8)
    if kind == 2:   # 노이즈 텍스처 (저해상도 노이즈 업샘플)
        small = rng.integers(0, 256, (max(h // 32, 2), max(w // 32, 2), 3), dtype=np.uint8)
        return np.array(Image.fromarray(small).resize((w, h), Image.BICUBIC))
    return np.zeros((h, w, 3), np.uint8)  # 검정(원본과 동일)

def composite(path, rng):
    im = Image.open(path).convert('RGBA')
    im.thumbnail((MAX_SIDE, MAX_SIDE))
    a = np.asarray(im).astype(np.float32)
    rgb, alpha = a[..., :3], a[..., 3:] / 255.0
    bg = random_background(im.width, im.height, rng).astype(np.float32)
    return Image.fromarray((rgb * alpha + bg * (1 - alpha)).astype(np.uint8))

if OUT_DIR.exists():
    shutil.rmtree(OUT_DIR)
for s in ('train', 'val', 'test'):
    rng = np.random.default_rng(SEED + hash(s) % 1000)
    for c in CLASSES:
        d = OUT_DIR / s / c; d.mkdir(parents=True, exist_ok=True)
        files = split_files[s][c]
        if s == 'train':
            reps = max(1, int(np.ceil(TARGET_PER_CLASS / len(files))))
            items = [(f, k) for k in range(reps) for f in files][:max(TARGET_PER_CLASS, len(files))]
        else:
            items = [(f, 0) for f in files]
        for f, k in items:
            composite(f, rng).save(d / f'{f.stem}_bg{k}.jpg', quality=95)

for s in ('train', 'val', 'test'):
    print(s, {c: len(list((OUT_DIR / s / c).glob('*.jpg'))) for c in CLASSES})

In [ ]:
# 샘플 확인
import matplotlib.pyplot as plt
fig, axes = plt.subplots(len(CLASSES), 6, figsize=(14, 2.6 * len(CLASSES)))
for i, c in enumerate(CLASSES):
    fs = random.sample(list((OUT_DIR / 'train' / c).glob('*.jpg')), 6)
    for j, f in enumerate(fs):
        axes[i, j].imshow(Image.open(f)); axes[i, j].axis('off')
    axes[i, 0].set_title(c, loc='left')
plt.tight_layout(); plt.show()

## 3. 학습 — YOLO11 vs YOLO26

- V100 은 FP16 텐서코어 지원 → AMP 기본 활성화 (bf16 / FlashAttention 은 미지원)
- 분류는 imgsz 224~320 이면 충분. 병변이 작으면 320 권장

In [ ]:
from ultralytics import YOLO

# 서버에 분류용 가중치가 없음(인터넷 차단): common/weights/ 에 업로드 후 실행
EXPERIMENTS = ['../common/weights/yolo11s-cls.pt', '../common/weights/yolo26s-cls.pt']   # 필요시 m 크기 추가
COMMON = dict(data=str(OUT_DIR), imgsz=320, epochs=60, batch=128, device=0, workers=8,
              patience=15, seed=SEED, project='runs/rose_cls', exist_ok=True,
              hsv_h=0.01, hsv_s=0.4, hsv_v=0.4, fliplr=0.5, flipud=0.5, degrees=30, erasing=0.3)

results = {}
for w in EXPERIMENTS:
    name = Path(w).stem
    model = YOLO(w)
    model.train(name=name, **COMMON)
    results[name] = Path('runs/rose_cls') / name / 'weights' / 'best.pt'
results

## 4. 테스트셋 평가

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

test_files = [(f, c) for c in CLASSES for f in sorted((OUT_DIR / 'test' / c).glob('*.jpg'))]
y_true = [CLASSES.index(c) for _, c in test_files]

for name, wpath in results.items():
    model = YOLO(wpath)
    names = model.names  # {idx: class}
    y_pred = []
    for i in range(0, len(test_files), 256):
        for r in model.predict([str(f) for f, _ in test_files[i:i + 256]], imgsz=320, device=0, verbose=False):
            y_pred.append(CLASSES.index(names[int(r.probs.top1)]))
    print(f'===== {name} =====')
    print(classification_report(y_true, y_pred, target_names=CLASSES, digits=3))
    ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=CLASSES).plot(
        xticks_rotation=30, cmap='Greens'); plt.title(name); plt.show()

### 다음 단계
- 결과가 좋은 모델을 Phase 3 의 2단계 파이프라인 분류기로 재사용
- Phase 2: 알파 마스크 기반 copy-paste 합성 → `yolo26s-seg` 잎 검출기 학습